In [ ]:
#Loading in Packages and Data

#Importing Packages
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr
import os; import time
import pickle
import h5py
###############################################################
def coefs(coefficients,degree):
    coef=coefficients
    coefs=""
    for n in range(degree, -1, -1):
        string=f"({coefficients[len(coef)-(n+1)]:.1e})"
        coefs+=string + f"x^{n}"
        if n != 0:
            coefs+=" + "
    return coefs
###############################################################

#Importing Model Data
check=False
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'

# dx = 1 km; Np = 1M; Nt = 5 min
data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_1km_1e6.nc') #***
parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_1km_1e6.nc') #***
res='1km'
Np_str='1e6'

# dx = 1km; Np = 50M
#Importing Model Data
check=False
dir2='/home/air673/koa_scratch/'
data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_50M.nc') #***
res='1km'; t_res='1min'; Np_str='50e6'

# # dx = 1km; Np = 100M
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_1km_1min.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_1km_1min_100M.nc') #***
# res='1km'; t_res='1min'; Np_str='100e6'


# dx = 250 m
# #Importing Model Data
# check=False
# dir2='/home/air673/koa_scratch/'
# data=xr.open_dataset(dir2+'cm1out_250m.nc') #***
# parcel=xr.open_dataset(dir2+'cm1out_pdata_250m.nc') #***

In [ ]:
def check_memory():
    import sys
    ipython_vars = ["In", "Out", "exit", "quit", "get_ipython", "ipython_vars"]
    print("Top 10 objects with highest memory usage")
    # Get a sorted list of the objects and their sizes
    mem = {
        key: round(value/1e6,2)
        for key, value in sorted(
            [
                (x, sys.getsizeof(globals().get(x)))
                for x in globals()
                if not x.startswith("_") and x not in sys.modules and x not in ipython_vars
            ],
            key=lambda x: x[1],
            reverse=True)[:10]
    }
    print({key:f"{value} MB" for key,value in mem.items()})
    print(f"\n{round(sum(mem.values()),2)/1000} GB in use overall")

In [ ]:
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
path=dir2+'../Functions/'
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions


# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

In [ ]:
#PROFILES AT SINGLE TIME (CLOUDY)

t=80

var='winterp';w=data[var].isel(time=t).data

w_z = np.mean(w,axis=(1,2))
mask1 = (data['winterp'][t] >= 0.5) & ((data['qc'][t] + data['qi'][t]) >= 1e-6)
print(np.any(mask1))
# masked_profile1 = np.ma.masked_where(~mask1, HMC)
masked_profile1 = np.where(~mask1, np.nan, w)
w_cloudy= np.nanmean(masked_profile1, axis=(1, 2))
print('done')

#########################


# New Subplots for Contour Plots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

#First plot
axes[0].plot(w_z,data['zh'])
axes[0].axvline(0,color='black')
axes[0].set_title('w(z) Horizontal Average for Eulerian Data Everywhere',fontsize=8)



#Second plot
axes[1].plot(w_cloudy,data['zh'])
axes[1].axvline(0,color='black')
axes[1].set_title('w(z) Horizontal Average for Eulerian Data (w ≥ 0.5) & (qc+qi ≥ 1e-6)',fontsize=8)


apply_scientific_notation([axes[0],axes[1]])
plt.suptitle(f'Horizontal Average of W For Entire Domain vs Within Cloudy Updrafts (AT TIME {t})')
# plt.tight_layout()

In [ ]:
#PROFILES AT SINGLE TIME (GENERAL)

t=80

var='winterp';w=data[var].isel(time=t).data

w_z = np.mean(w,axis=(1,2))
mask1 = (data['winterp'][t] >= 0.1) & ((data['qc'][t] + data['qi'][t]) < 1e-6)
print(np.any(mask1))
# masked_profile1 = np.ma.masked_where(~mask1, HMC)
masked_profile1 = np.where(~mask1, np.nan, w)
w_cloudy= np.nanmean(masked_profile1, axis=(1, 2))
print('done')

#########################

# New Subplots for Contour Plots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

#First plot
axes[0].plot(w_z,data['zh'])
axes[0].axvline(0,color='black')
axes[0].set_title('w(z) Horizontal Average for Eulerian Data Everywhere',fontsize=8)



#Second plot
axes[1].plot(w_cloudy,data['zh'])
axes[1].axvline(0,color='black')
axes[1].set_title('w(z) Horizontal Average for Eulerian Data (w ≥ 0.5) & (qc+qi ≥ 1e-6)',fontsize=8)


apply_scientific_notation([axes[0],axes[1]])
plt.suptitle(f'Horizontal Average of W For Entire Domain vs Within General Updrafts (AT TIME {t})')
# plt.tight_layout()

In [ ]:
# Reading Back Data Later
##############
def make_data_dict(in_file,var_names,read_type):
    if read_type=='h5py':
        with h5py.File(in_file, 'r') as f:
            data_dict = {var_name: f[var_name][:] for var_name in var_names}
            
    elif read_type=='xarray':
        in_data = xr.open_dataset(
            in_file,
            engine='h5netcdf',
            phony_dims='sort',
            chunks={'phony_dim_0': 100, 'phony_dim_1': 1_000_000} 
        )
        data_dict = {k: in_data[k][:].compute().data for k in var_names}
    return data_dict

# read_type='xarray'
read_type='h5py'

In [ ]:
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
in_file=dir2+f'lagrangian_binary_array_{res}_{t_res}_{Np_str}.h5'

var_names = ['A_g', 'A_c', 'W', 'Z', 'Y', 'X']
data_dict = make_data_dict(in_file,var_names,read_type)
A_g, A_c, W, Z, Y, X = (data_dict[k] for k in var_names)

# #Making Time Matrix
Nt=len(data['time'])
T = np.broadcast_to(np.arange(Nt)[:, None], A_c.shape)  # shape: (Nt, p)

check_memory()

In [ ]:
#READING BACK IN
dir2=dir+'Project_Algorithms/Entrainment/'
in_file=dir2+f'processed_binary_arrays_{res}_{t_res}_{Np_str}.h5'

var_names = ['A_g_Processed', 'A_c_Processed']
data_dict = make_data_dict(in_file,var_names,read_type)
A_g_Processed, A_c_Processed = (data_dict[k] for k in var_names)
check_memory(globals())

In [ ]:
#############################################################################
#############################################################################

In [ ]:
def VMF2d(A, T, Z):
    start_time = time.time()
    """
    Function to compute 2D Mass Flux and update result array based on provided inputs.
    
    Returns a 2D (t,z) array containing the sum of the D array represented by parcels in cloudy updrafts by 1.
    The finally array is then ordered by the appropiate index using the np.add.at function.
    
    Parameters:
    - A: The (t,p) lagrangian binary array.
    - T: The (t,p) lagrangian time index array.
    - Z: The (t,p) Lagrangian z index array.

    """
    # Compute the difference between neighboring elements along the first axis
    D = A * W
    
    # # Update D for entrainment/detrainment
    # if type=='e':
    #     D[D <= 0] = 0
    # elif type=='d':
    #     D[D >= 0] = 0
    
    # Initialize time and vertical dimension arrays
    Nt = len(data['time']); Nz = len(data['zh'])
    
    # Initialize result array
    result = np.zeros((Nt, Nz))
    
    # Use np.add.at to accumulate values in the result array
    np.add.at(result, (T, Z), D)
    
    end_time = time.time()
    print(f"Execution time: {(end_time - start_time)} seconds")
    return result

In [ ]:
#TESTING
Ws=W[np.where(A_c==1)]
plt.hist(Ws,bins=100);
plt.xscale('log')
plt.ylabel('count');plt.xlabel('W (m/s)')
plt.title('Histogram of all W values used in Cloudy VMF calculation')
print(W[np.where(A_c==1)].mean())
print(np.percentile(W[np.where(A_c==1)],95))

In [ ]:
#TESTING
Ws=W[np.where(A_g==1)]
plt.hist(Ws,bins=100);
plt.xscale('log')
plt.ylabel('count');plt.xlabel('W (m/s)')
plt.title('Histogram of all W values used in General VMF calculation')
print(W[np.where(A_g==1)].mean())
print(np.percentile(W[np.where(A_g==1)],95))

In [ ]:
# len(np.where(A_c==1)[0]),len(np.where(A_g==1)[0])
# ~700e3 vs ~15e6

In [ ]:
#TURN PROCESSING ON OR OFF
PROCESSING=False
PROCESSING=True

# Set A based on PROCESSING state
print('Calculating 2D VMF for General Updrafts')
A = A_g if (PROCESSING==False) else A_g_Processed
profile_array_VMF_g = VMF2d(A, T, Z)


# Set A for the second block
print('Calculating 2D VMF for Cloudy Updrafts')
A = A_c if (PROCESSING==False) else A_c_Processed
profile_array_VMF_c = VMF2d(A, T, Z)


#SAVING
if PROCESSING==False:
    dir3=dir+f'Project_Algorithms/Entrainment/2D_VMF_profiles_{res}_{t_res}_{Np_str}.h5'
if PROCESSING==True:
    dir3=dir+f'Project_Algorithms/Entrainment/2D_VMF_profiles_PREPROCESSING_{res}_{t_res}_{Np_str}.h5'
with h5py.File(dir3, "w") as h5f:
    h5f.create_dataset("profile_array_VMF_g", data=profile_array_VMF_g)
    h5f.create_dataset("profile_array_VMF_c", data=profile_array_VMF_c)
print('done')

In [ ]:
########################################################
#PLOTTING

In [ ]:
#constants
Cp=1004 #Jkg-1K-1
Cv=717 #Jkg-1K-1
Rd=Cp-Cv #Jkg-1K-1
eps=0.608

Lx=(data['xf'][-1].item()-data['xf'][0].item())*1000 #x length (m)
Ly=(data['yf'][-1].item()-data['yf'][0].item())*1000 #y length (m)
Np=len(parcel['xh']) #number of lagrangian parcles
dt=(data['time'][1]-data['time'][0]).item()/1e9 #sec
dx=(data['xf'][1].item()-data['xf'][0].item())*1e3 #meters
dy=(data['yf'][1].item()-data['yf'][0].item())*1e3 #meters
xs=data['xf'].values*1000
ys=data['yf'].values*1000
zs=data['zf'].values*1000

def zf(z):
    k=z #z is the # level of z
    out=data['zf'].values[k]*1000
    
    return out
    
# def rho(x,y,z,t):
#     p=data['prs'].isel(xh=x,yh=y,zh=z,time=t).item()
#     p0=101325 #Pa
#     theta=data['th'].isel(xh=x,yh=y,zh=z,time=t).item()
#     T=theta*(p/p0)**(Rd/Cp)
#     qv=data['qv'].isel(xh=x,yh=y,zh=z,time=t).item()
#     # Tv=T*(1+eps*qv)
#     Tv=T*(eps+qv)/(eps*(1+qv))
#     rho = p/(Rd*Tv)
#     out=rho
#     return out

def rho(x,y,z,rho_data_t):
    out=rho_data_t[z,y,x]
    return out
def m(t):
    rho_data_t=data['rho'].isel(time=t).data
    
    m=0
    #triple sum
    for k in range(len(data['zh'])):
        dz=(zf(k+1)-zf(k))
        for j in range(len(data['yh'])):
            for i in range(len(data['xh'])):
                rho_out=rho(i,j,k,rho_data_t)
                m+=rho_out*dz
                
    #triple sum
    out=m*dx*dy/Np
    return out


In [ ]:
if plotting==True:
    #Calculate Mass Constant
    # calculate='single_time'
    # calculate=True
    calculate=False
    
    if calculate==True:
        Nt=len(data['time'])
        m_arr=np.zeros((Nt))
        for t in np.arange(Nt):
            if np.mod(t,25)==0: print(t)
            m_arr[t]=m(t)
        np.save(f'Mass_Array_{res}_{t_res}_{Np_str}.npy', m_arr)
    elif calculate=='single_time':
        Nt=len(data['time'])
        m_arr=np.zeros((Nt))
    
        t=len(data['time'])//2 #Pick some middle time
        m_300=m(t)
        for t in np.arange(Nt):
            m_arr[t]=m_300 #UNCOMMENT FOR FULL CALCULATION
        # np.save('Mass_Array.npy', m_arr)
        np.save(f'Mass_Array_{res}_{t_res}_{Np_str}.npy', m_arr)
    else:
        dir3=dir+f'Project_Algorithms/Entrainment/'
        # m_arr = np.load('Mass_Array.npy')
        m_arr = np.load(f'Mass_Array_{res}_{t_res}_{Np_str}.npy')
    
    # # TESTING
    # lst=[]
    # for t in np.arange(133):
    #     lst.append(m_arr[t])
    
    # plt.plot(lst)
    # (np.max(lst)-np.min(lst))*100/np.mean(lst)

In [ ]:
PROCESSING=False
PROCESSING=True

if PROCESSING==False:
    dir3=dir+f'Project_Algorithms/Entrainment/2D_VMF_profiles_{res}_{t_res}_{Np_str}.h5'
if PROCESSING==True:
    dir3=dir+f'Project_Algorithms/Entrainment/2D_VMF_profiles_PREPROCESSING_{res}_{t_res}_{Np_str}.h5'
with h5py.File(dir3, "r") as h5f:
    profile_array_VMF_g = h5f["profile_array_VMF_g"][:]
    profile_array_VMF_c = h5f["profile_array_VMF_c"][:]

In [ ]:
def apply_constant(profile_array,apply):
    if apply==True:
        
        Nt=profile_array.shape[0]
        Nz=profile_array.shape[1]
        # print(Lx,Ly)
        profile_array/=(Lx*Ly)
        for t in np.arange(Nt):
            profile_array[t]*=m_arr[t]
            # print(m_arr[t])
        for z in np.arange(Nz):
            dz=zf(z+1)-zf(z)
            profile_array[:,z]/=dz
            # print(dz)
    return profile_array

profile_array_VMF_g=apply_constant(profile_array_VMF_g,apply=True)
profile_array_VMF_c=apply_constant(profile_array_VMF_c,apply=True)

In [ ]:
type='general'
# type='cloudy'

if type=='general':
    profile_array=profile_array_VMF_g.copy()
if type=='cloudy':
    profile_array=profile_array_VMF_c.copy()


import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np

fig = plt.figure(figsize=(10, 8))
gs = GridSpec(2, 2, figure=fig)

######
cmap1 = plt.cm.viridis
cmap2 = plt.cm.seismic 
n_levels=29
######

# ######
# vmax_shared = np.max([np.max(profile_array_e), np.max(profile_array_d)])
# norm_shared = mcolors.Normalize(vmin=0, vmax=vmax_shared)
# ######

# First subplot: VMF
########################################
ax1 = fig.add_subplot(gs[0, 0])
contour1 = ax1.contourf(profile_array.T, cmap=cmap1)
cbar1=fig.colorbar(contour1, ax=ax1)
apply_scientific_notation_colorbar([cbar1])
Nz = len(data['zh'])
ax1.set_yticks(np.arange(Nz))
new_ytick_labels = np.round(data['zf'].values[:Nz], 2)
ax1.set_yticklabels(new_ytick_labels, fontsize=8, rotation=0)
ax1.set_ylabel('z (km)');ax1.set_xlabel('t (timesteps)')
ax1.set_title('Entrainment using Lagrangian Binary Array',fontsize=8)

# #FIXING TICKS
# ax3.set_yticks(np.arange(Nz))
# new_ytick_labels = np.round(data['zf'].values[:Nz], 2)
# ax3.set_yticklabels(new_ytick_labels, fontsize=8, rotation=0)
# ax3.set_ylabel('z (km)');ax3.set_xlabel('t (timesteps)')
# ax3.set_title('Entrainment - Detrainment')

# #FIXING SCIENTIFIC NOTATION
# from matplotlib.ticker import ScalarFormatter
# formatter = ScalarFormatter(useMathText=True)
# formatter.set_powerlimits((-2, 2))  # Adjust the range for scientific notation
# for cbar in (cbar1,cbar2, cbar3):  # These must be Colorbar instances
#     cbar.formatter = formatter
#     cbar.update_ticks()

# Display the plot
plt.tight_layout()

In [ ]:
plt.plot(np.mean(profile_array,axis=(0)),data['zh'],label='VMF')

plt.legend(); plt.title('2D Vertical Mass Flux Using Lagrangian Binary Array')

from matplotlib.ticker import ScalarFormatter
formatter = ScalarFormatter(useMathText=True)
formatter.set_scientific(True)
formatter.set_powerlimits((-1, 1))
plt.gca().xaxis.set_major_formatter(formatter)

plt.ylabel('z (km)');plt.xlabel('Vertical Mass Flux, M(t,z) ' + r'$(kg m^{-2} s^{-1})$')

# #COMPARING MAXIMUM WITH ROMPS 2012
# (1.4e-2)/(4e-3)